# Topic 1: RDD Fundamentals

> **Phase 2 → Week 2 → Topic 1**
>
> Prerequisites: Week 1 complete (Architecture, DAG, Transformations vs Actions)

---

## Why This Topic Matters for Interviews

RDDs are the **foundation of Spark**. Every DataFrame and Dataset is built on top of RDDs under the hood. Even if you never write RDD code at work, interviewers ask:
- "What is an RDD?"
- "What are the 5 properties of an RDD?"
- "What is immutability and why does Spark use it?"
- "How is partitioning related to parallelism?"

You cannot answer these without understanding RDDs.

---

## The Spreadsheet Tabs Analogy

Imagine a Google Sheet with 1 million rows of sales data:
- You split it into **tabs** (Jan, Feb, Mar ... Dec) → these are **partitions**
- Each tab lives on a **different computer** → distributed storage
- You **cannot edit a tab in place** — you must create a new sheet → **immutability**
- The sheet knows its **history** (version history) → **lineage**
- Any tab can be **rebuilt from the original CSV** if deleted → **fault tolerance**

That's an RDD.

---

## What is an RDD?

**RDD = Resilient Distributed Dataset**

Break it down:

| Word | Meaning |
|------|--------|
| **Resilient** | Fault-tolerant — lost partitions can be recomputed from lineage |
| **Distributed** | Data is split across multiple machines (partitions) |
| **Dataset** | A collection of data records |

Formally: an RDD is a **read-only, partitioned collection of records** that can be
operated on in parallel across a cluster.

### The 5 Key Properties of Every RDD

Every RDD internally tracks these 5 things:

```
1. getPartitions()      → list of partitions (how data is split)
2. compute(partition)   → how to compute data for a given partition
3. getDependencies()    → parent RDDs this RDD depends on (lineage)
4. getPreferredLocations() → which nodes hold the data (data locality)
5. partitioner          → how keys are distributed (for key-value RDDs)
```

When you write `rdd.filter(lambda x: x > 0)`, Spark creates a new RDD with:
- Same partition count as parent
- `compute()` = parent's compute + the filter function
- `getDependencies()` = points to the parent RDD

---

## 3 Ways to Create an RDD

### Method 1: `parallelize()` — from a Python collection
```python
sc.parallelize([1, 2, 3, 4, 5])           # Spark decides partition count
sc.parallelize([1, 2, 3, 4, 5], 3)        # explicitly 3 partitions
```
Use for: testing, small data, learning.

### Method 2: `textFile()` — from a file on disk/HDFS/S3
```python
sc.textFile("data.txt")                   # each line = one element
sc.textFile("s3://bucket/logs/*.log")     # wildcards supported
sc.textFile("hdfs://namenode/data/")      # HDFS directory
```
Use for: reading raw text/CSV files, log files.

### Method 3: From another RDD (via transformation)
```python
rdd2 = rdd1.filter(lambda x: x > 0)      # transformation returns new RDD
rdd3 = rdd2.map(lambda x: x * 2)
```
The most common way — chain transformations.

---

## Partitions — The Unit of Parallelism

```
RDD with 4 partitions across 4 workers:

  RDD
  ├── Partition 0: [1, 2, 3, 4]        → Worker Node 1, Executor 1
  ├── Partition 1: [5, 6, 7, 8]        → Worker Node 2, Executor 2
  ├── Partition 2: [9, 10, 11, 12]     → Worker Node 3, Executor 3
  └── Partition 3: [13, 14, 15, 16]    → Worker Node 4, Executor 4

  All 4 executors process their partition simultaneously!
```

**Key rules:**
- 1 partition = 1 task (per stage)
- More partitions = more parallelism (up to executor core count)
- `getNumPartitions()` → check partition count
- `glom().collect()` → see what each partition contains

**Default partition count:**
- `parallelize()` → `sc.defaultParallelism` (usually = total executor cores)
- `textFile()` → number of HDFS blocks (128MB each by default)
- After shuffle → `spark.sql.shuffle.partitions` (default: 200)

---

## Immutability — You Cannot Change an RDD

RDDs are **read-only**. Every transformation creates a **new RDD**.

```python
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = rdd1.filter(lambda x: x > 2)    # new RDD — rdd1 unchanged
rdd3 = rdd2.map(lambda x: x * 10)      # new RDD — rdd2 unchanged

# rdd1 still has [1, 2, 3, 4, 5]
# rdd2 still has [3, 4, 5]
# rdd3 has [30, 40, 50]
```

**Why immutability?**
1. **Fault tolerance**: lineage is reliable only if parent data never changes
2. **Parallelism**: no locking needed if data can't be modified
3. **Reproducibility**: recomputing a partition always gives the same result

---

## Lazy Evaluation (Quick Recap)

Transformations on RDDs are **lazy** — they build the DAG but don't run.
Actions trigger execution.

```
rdd.filter()  → lazy  → adds to DAG
rdd.map()     → lazy  → adds to DAG
rdd.count()   → EAGER → triggers execution of entire DAG
```

---

## Interview Cheat Sheet

**Q: What does RDD stand for and what does each word mean?**
> Resilient (fault-tolerant via lineage), Distributed (split across machines), Dataset (a collection of data records).

**Q: What are the 5 properties of an RDD?**
> 1. A list of partitions. 2. A compute function per partition. 3. Dependencies on parent RDDs (lineage). 4. Optional preferred locations per partition (data locality). 5. Optional partitioner for key-value RDDs.

**Q: How is an RDD different from a Python list?**
> A Python list lives on one machine in memory. An RDD is split into partitions distributed across many machines, processed in parallel, fault-tolerant via lineage, and lazily evaluated. An RDD of 100GB can exist across 100 machines; a Python list of 100GB would crash your laptop.

**Q: What is immutability in Spark and why does it matter?**
> RDDs cannot be modified — every transformation creates a new RDD. This makes fault tolerance reliable (recomputing a partition always gives the same result) and eliminates concurrency issues (no locks needed).

**Q: How does Spark decide the number of partitions?**
> For `parallelize()`: defaults to `sc.defaultParallelism` (= total executor cores). For file reads: determined by the number of HDFS/S3 blocks (128MB default). After a shuffle: determined by `spark.sql.shuffle.partitions` (default 200). You can also specify explicitly.

**Q: What is the relationship between partitions and parallelism?**
> Each partition processes as one task, one executor core slot. Total parallel tasks = number of executor cores. More partitions than cores = multiple waves of tasks. Fewer partitions than cores = idle cores. Optimal is ~2-4× the number of executor cores.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week2 - RDD Fundamentals") \
    .master("local[4]") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Default parallelism (= cores): {sc.defaultParallelism}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/14 03:07:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
Default parallelism (= cores): 4


## Part 1: Creating RDDs — 3 Methods

In [2]:
# Method 1: sc.parallelize() — from a Python list
rdd1 = sc.parallelize([10, 20, 30, 40, 50, 60, 70, 80])  # Spark picks partition count
rdd2 = sc.parallelize([10, 20, 30, 40, 50, 60, 70, 80], 4)  # explicit 4 partitions

print("=== Method 1: parallelize ===")
print(f"rdd1 partitions (auto): {rdd1.getNumPartitions()}")
print(f"rdd2 partitions (explicit 4): {rdd2.getNumPartitions()}")
print(f"rdd2 data: {rdd2.collect()}")

=== Method 1: parallelize ===
rdd1 partitions (auto): 4
rdd2 partitions (explicit 4): 4


rdd2 data: [10, 20, 30, 40, 50, 60, 70, 80]


In [3]:
# Method 2: sc.parallelize() with a range (common for testing)
rdd_range = sc.parallelize(range(1, 101), 4)  # 1 to 100, 4 partitions
print("=== Method 2: parallelize with range ===")
print(f"Partitions: {rdd_range.getNumPartitions()}")
print(f"First 10: {rdd_range.take(10)}")
print(f"Count: {rdd_range.count()}")

=== Method 2: parallelize with range ===
Partitions: 4


First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


Count: 100


In [4]:
# Method 3: From transformation (most common in practice)
rdd_base = sc.parallelize(range(1, 21), 4)
rdd_filtered = rdd_base.filter(lambda x: x % 2 == 0)   # new RDD
rdd_squared = rdd_filtered.map(lambda x: x ** 2)        # another new RDD

print("=== Method 3: From Transformation ===")
print(f"Base RDD: {rdd_base.collect()}")
print(f"Filtered (evens): {rdd_filtered.collect()}")
print(f"Squared evens: {rdd_squared.collect()}")

=== Method 3: From Transformation ===


Base RDD: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


Filtered (evens): [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


Squared evens: [4, 16, 36, 64, 100, 144, 196, 256, 324, 400]


## Part 2: Inspecting Partitions — `glom()` and `getNumPartitions()`

`glom()` is a special transformation that groups each partition into a list.
It's the best way to SEE how data is distributed across partitions.

```
Before glom():  RDD  →  [1, 2, 3, 4, 5, 6, 7, 8]  (4 partitions)
After glom():   RDD  →  [[1,2], [3,4], [5,6], [7,8]]  (each element = one partition)
```

In [5]:
# See how data is distributed across partitions using glom()
rdd = sc.parallelize(range(1, 17), 4)  # 16 numbers, 4 partitions

partitions = rdd.glom().collect()
print(partitions)

print("Data distribution across partitions:")
for i, partition in enumerate(partitions):
    print(f"  Partition {i}: {partition}  ({len(partition)} elements)")

print(f"\nTotal elements: {rdd.count()}")
print(f"Num partitions: {rdd.getNumPartitions()}")

[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
Data distribution across partitions:
  Partition 0: [1, 2, 3, 4]  (4 elements)
  Partition 1: [5, 6, 7, 8]  (4 elements)
  Partition 2: [9, 10, 11, 12]  (4 elements)
  Partition 3: [13, 14, 15, 16]  (4 elements)



Total elements: 16
Num partitions: 4


In [6]:
# Try different partition counts — see how data is distributed
for num_parts in [1, 2, 4, 8]:
    rdd = sc.parallelize(range(1, 17), num_parts)
    partitions = rdd.glom().collect()
    sizes = [len(p) for p in partitions]
    print(f"Partitions: {num_parts:2d}  → sizes: {sizes}  → parallel tasks: {num_parts}")

Partitions:  1  → sizes: [16]  → parallel tasks: 1


Partitions:  2  → sizes: [8, 8]  → parallel tasks: 2


Partitions:  4  → sizes: [4, 4, 4, 4]  → parallel tasks: 4


Partitions:  8  → sizes: [2, 2, 2, 2, 2, 2, 2, 2]  → parallel tasks: 8


In [7]:
# Strings — common in real RDDs (log processing)
log_lines = [
    "2024-01-01 ERROR: Connection timeout",
    "2024-01-01 INFO: User logged in",
    "2024-01-02 ERROR: Disk full",
    "2024-01-02 WARN: High memory usage",
    "2024-01-03 INFO: Backup completed",
    "2024-01-03 ERROR: Database down",
]

log_rdd = sc.parallelize(log_lines, 3)
print(f"Log RDD — {log_rdd.getNumPartitions()} partitions")
print(f"First entry: {log_rdd.first()}")
print(f"Count: {log_rdd.count()}")

Log RDD — 3 partitions
First entry: 2024-01-01 ERROR: Connection timeout


Count: 6


## Part 3: Immutability — RDDs Never Change

In [8]:
# Prove immutability: transformations return NEW RDDs
original = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 2)

filtered = original.filter(lambda x: x > 5)     # new RDD
doubled  = original.map(lambda x: x * 2)         # another new RDD (from original!)

print("=== Immutability Demo ===")
print(f"original : {original.collect()}")
print(f"filtered : {filtered.collect()}")
print(f"doubled  : {doubled.collect()}")
print(f"original unchanged: {original.collect()}")
print()
print("All 3 are DIFFERENT RDD objects:")
print(f"  id(original) = {id(original)}")
print(f"  id(filtered) = {id(filtered)}")
print(f"  id(doubled)  = {id(doubled)}")

=== Immutability Demo ===
original : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


filtered : [6, 7, 8, 9, 10]
doubled  : [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


original unchanged: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

All 3 are DIFFERENT RDD objects:
  id(original) = 140058447053392
  id(filtered) = 140057864019200
  id(doubled)  = 140058423584032


## Part 4: Lineage — The RDD Family Tree

`toDebugString()` shows the lineage chain — how an RDD was derived.

**Reading the output:**
- Read **bottom to top** (bottom = oldest ancestor)
- `(N)` = number of partitions
- Each `+` level = one transformation step
- `ShuffledRDD` = a shuffle happened here (stage boundary)

In [9]:
# Build a chain and inspect the lineage
rdd_a = sc.parallelize(range(100), 4)           # source
rdd_b = rdd_a.map(lambda x: x * 2)              # narrow
rdd_c = rdd_b.filter(lambda x: x > 50)          # narrow
rdd_d = rdd_c.map(lambda x: (x % 5, x))         # narrow → key-value pairs
rdd_e = rdd_d.reduceByKey(lambda a, b: a + b)   # WIDE (shuffle!)

print("=== Lineage Graph (toDebugString) ===")
print("Read BOTTOM to TOP:")
print()
print(rdd_e.toDebugString().decode())
print()
print("Explanation:")
print("  PythonRDD (bottom) = sc.parallelize — the source")
print("  Each PythonRDD above = one narrow transformation")
print("  ShuffledRDD = the reduceByKey shuffle boundary")
print("  The top PythonRDD = rdd_e (our final result)")

=== Lineage Graph (toDebugString) ===
Read BOTTOM to TOP:

(4) PythonRDD[31] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[30] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[29] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[28] at reduceByKey at /tmp/ipykernel_17/2573975503.py:6 []
    |  PythonRDD[27] at reduceByKey at /tmp/ipykernel_17/2573975503.py:6 []
    |  ParallelCollectionRDD[26] at readRDDFromFile at PythonRDD.scala:289 []

Explanation:
  PythonRDD (bottom) = sc.parallelize — the source
  Each PythonRDD above = one narrow transformation
  ShuffledRDD = the reduceByKey shuffle boundary
  The top PythonRDD = rdd_e (our final result)


In [10]:
# Fault tolerance simulation:
# If a partition of rdd_c is lost, Spark knows:
# rdd_c = rdd_b.filter()  AND  rdd_b = rdd_a.map()  AND  rdd_a = parallelize(range(100))
# So Spark re-runs: parallelize → map → filter for JUST that partition

# This is automatic — you never see it happen. But it's why you don't need
# intermediate disk writes like MapReduce.

result = rdd_e.collect()
print("Final result of rdd_e (reduceByKey sum, key = x%5):")
print(sorted(result))

Final result of rdd_e (reduceByKey sum, key = x%5):
[(0, 1750), (1, 1890), (2, 1830), (3, 1920), (4, 1860)]


## Part 5: Data Locality — Spark Tries to Process Data Where it Lives

Spark's `getPreferredLocations()` tells the scheduler which executor/node
prefers to process each partition (because the data is already there — no network transfer).

- With HDFS: each block has a preferred node (where it's stored)
- With S3: all nodes are equally good (S3 is remote for everyone)
- With `parallelize()`: Spark distributes randomly, then tasks run wherever

You don't manage this — Spark does it automatically. But it's WHY Spark is
faster with HDFS than S3 for some workloads (local reads vs network reads).

## Part 6: RDD vs DataFrame — When to Use Which?

| Feature | RDD | DataFrame |
|---------|-----|----------|
| API level | Low-level (functional) | High-level (declarative) |
| Optimization | Manual (you write efficient code) | Catalyst Optimizer does it |
| Type safety | Yes (Python types) | Schema-based |
| Best for | Custom transformations, ML pipelines, unstructured data | Structured/semi-structured data, SQL, ETL |
| Performance | Slower (no Catalyst, no Tungsten) | Faster (Catalyst + Tungsten optimizations) |
| When created | Spark 1.0 (2014) | Spark 1.3 (2015) |

**In practice (2024):** Use DataFrames for 95% of work. Learn RDDs because:
1. Interviews always ask about them
2. Understanding RDDs = understanding how DataFrames work internally
3. Some transformations (mapPartitions, custom serialization) still need RDDs
4. MLlib pipelines use RDDs internally

## Exercises — Try These Yourself

1. Create an RDD of the first 50 Fibonacci numbers with 5 partitions. Use `glom()` to see distribution.
2. Create an RDD from `range(1, 1001)`. Apply 3 transformations (your choice). Check the lineage with `toDebugString()`.
3. Create two RDDs (rdd_a from range(1,11), rdd_b from rdd_a). Prove rdd_a is unchanged after creating rdd_b.
4. Create an RDD with 1 partition. Then create the same data with 8 partitions. Is `count()` result different? Is the data different?

In [11]:
# Your exercises here

# Exercise 1: Fibonacci RDD
def fibonacci(n):
    fibs = [0, 1]
    for _ in range(n - 2):
        fibs.append(fibs[-1] + fibs[-2])
    return fibs[:n]

fib_rdd = sc.parallelize(fibonacci(50), 5)
print("Exercise 1 — Fibonacci distribution:")
for i, p in enumerate(fib_rdd.glom().collect()):
    print(f"  Partition {i}: {len(p)} elements, first={p[0]}, last={p[-1]}")

Exercise 1 — Fibonacci distribution:


  Partition 0: 10 elements, first=0, last=34
  Partition 1: 10 elements, first=55, last=4181
  Partition 2: 10 elements, first=6765, last=514229
  Partition 3: 10 elements, first=832040, last=63245986
  Partition 4: 10 elements, first=102334155, last=7778742049


In [12]:
# Exercise 4: Partition count affects parallelism, NOT the data itself
rdd_1p = sc.parallelize(range(1, 11), 1)
rdd_8p = sc.parallelize(range(1, 11), 8)

print("Exercise 4 — Same data, different partition counts:")
print(f"  1 partition  → count={rdd_1p.count()}, data={rdd_1p.collect()}")
print(f"  8 partitions → count={rdd_8p.count()}, data={rdd_8p.collect()}")
print()
print("Same data! Partition count only affects parallelism, not the result.")
print("But with local[4] and 8 partitions: 2 waves of 4 tasks vs 1 task with 1 partition.")

Exercise 4 — Same data, different partition counts:


  1 partition  → count=10, data=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


  8 partitions → count=10, data=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Same data! Partition count only affects parallelism, not the result.
But with local[4] and 8 partitions: 2 waves of 4 tasks vs 1 task with 1 partition.
